In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DATA_PATH = r"C:\Users\soori\.cache\kagglehub\competitions\dmt-2026-2nd-assignment"

TRAIN_PATH = "/kaggle/input/competitions/dmt-2026-2nd-assignment/training_set_VU_DM.csv"
TEST_PATH  = "/kaggle/input/competitions/dmt-2026-2nd-assignment/test_set_VU_DM.csv"

SUBMISSION_DIR  = "/kaggle/working/"
SUBMISSION_PATH = os.path.join(SUBMISSION_DIR, "submission.csv")

os.makedirs(SUBMISSION_DIR, exist_ok=True)

# RELEVANCE LABELS

def assign_relevance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["relevance"] = 0
    df.loc[df["click_bool"]   == 1, "relevance"] = 1
    df.loc[df["booking_bool"] == 1, "relevance"] = 5
    return df

# FEATURE ENGINEERING

def add_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Date ────────────────────────────────────────────────
    df["date_time"]       = pd.to_datetime(df["date_time"])
    df["search_month"]    = df["date_time"].dt.month
    df["search_day"]      = df["date_time"].dt.dayofweek
    df["search_hour"]     = df["date_time"].dt.hour         

    # ── Travel party ────────────────────────────────────────
    df["total_people"]    = df["srch_adults_count"] + df["srch_children_count"]
    df["is_family"]       = (df["srch_children_count"] > 0).astype(int)
    df["is_solo"]         = (df["srch_adults_count"] == 1) & (df["srch_children_count"] == 0)
    df["is_solo"]         = df["is_solo"].astype(int)
    df["is_couple"]       = (df["srch_adults_count"] == 2) & (df["srch_children_count"] == 0)
    df["is_couple"]       = df["is_couple"].astype(int)
    df["is_group"]        = (df["srch_adults_count"] > 2).astype(int)
    df["people_per_room"] = df["total_people"] / df["srch_room_count"].replace(0, 1)

    # ── Trip style ───────────────────────────────────────────
    df["is_long_stay"]    = (df["srch_length_of_stay"] >= 4).astype(int)
    df["is_weekend_trip"] = df["srch_saturday_night_bool"].astype(int)
    df["is_last_minute"]  = (df["srch_booking_window"] <= 3).astype(int)
    df["is_planned"]      = (df["srch_booking_window"] > 14).astype(int)
    df["log_booking_win"] = np.log1p(df["srch_booking_window"])
    df["log_length_stay"] = np.log1p(df["srch_length_of_stay"])

    # ── Visitor history ──────────────────────────────────────
    df["has_hist_star"]       = df["visitor_hist_starrating"].notnull().astype(int)
    df["has_hist_price"]      = df["visitor_hist_adr_usd"].notnull().astype(int)
    df["is_high_end_user"]    = (df["visitor_hist_starrating"] >= 4).astype(int)
    df["star_pref_delta"]     = (
        df["prop_starrating"] - df["visitor_hist_starrating"]
    )  # positive = above their usual standard
    df["price_pref_delta"]    = (
        df["price_usd"] - df["visitor_hist_adr_usd"]
    )

    # ── Geographic ──────────────────────────────────────────
    df["same_country"]        = (
        df["visitor_location_country_id"] == df["prop_country_id"]
    ).astype(int)

    # ── Price ────────────────────────────────────────────────
    df["price_per_person"]    = df["price_usd"] / df["total_people"].replace(0, 1)
    df["price_per_night"]     = df["price_usd"] / df["srch_length_of_stay"].replace(0, 1)
    df["log_price"]           = np.log1p(df["price_usd"])

    # ── Property quality ─────────────────────────────────────
    df["quality_score"]       = (
        df["prop_starrating"] * 0.4 + df["prop_review_score"] * 0.6
    )  # weighted blend used as a single quality axis
    df["has_promotion"]       = df["promotion_flag"].fillna(0).astype(int)

    # ── Expedia positioning signals ──────────────────────────
    df["log_position"]        = np.log1p(df.get("position", 0))
    # position is only available in training; harmless NaN in test

    # ── Per-query relative ranks (competitive context) ───────
    for col, ascending in [
        ("price_usd",         True),
        ("prop_starrating",   False),
        ("prop_review_score", False),
        ("prop_location_score1", False),
    ]:
        if col in df.columns:
            df[f"{col}_rank"] = df.groupby("srch_id")[col].rank(
                ascending=ascending, method="average"
            )

    # ── Per-query z-score deviations ─────────────────────────
    for col in ["price_usd", "prop_review_score", "prop_starrating"]:
        if col in df.columns:
            grp  = df.groupby("srch_id")[col]
            mean = grp.transform("mean")
            std  = grp.transform("std").replace(0, 1)
            df[f"{col}_zscore"] = (df[col] - mean) / std
            df[f"{col}_diff"]   = df[col] - mean

    # ── Percentile within query ──────────────────────────────
    df["price_pct_rank"] = df.groupby("srch_id")["price_usd"].rank(pct=True)

    return df


def add_property_agg_features(train: pd.DataFrame, test: pd.DataFrame) -> tuple:
    """
    Compute historical booking/click rates per property from training data.
    Avoids leakage by computing on the full training set before the split.
    """
    # Click-through and booking rates per property
    prop_stats = (
        train.groupby("prop_id")
        .agg(
            prop_click_rate  = ("click_bool",   "mean"),
            prop_book_rate   = ("booking_bool", "mean"),
            prop_n_queries   = ("srch_id",      "nunique"),
            prop_mean_price  = ("price_usd",    "mean"),
            prop_mean_star   = ("prop_starrating", "mean"),
            prop_mean_review = ("prop_review_score", "mean"),
        )
        .reset_index()
    )

    # Country-level booking rates
    country_stats = (
        train.groupby("prop_country_id")
        .agg(
            country_book_rate = ("booking_bool", "mean"),
            country_click_rate = ("click_bool",  "mean"),
        )
        .reset_index()
    )

    train = train.merge(prop_stats,    on="prop_id",         how="left")
    train = train.merge(country_stats, on="prop_country_id", how="left")

    test  = test.merge(prop_stats,    on="prop_id",         how="left")
    test  = test.merge(country_stats, on="prop_country_id", how="left")

    # Fill unseen properties (cold start) with a conservative low value
    for df in [train, test]:
        for col in ["prop_click_rate", "prop_book_rate"]:
            df[col] = df[col].fillna(df[col].quantile(0.25))
        for col in ["country_book_rate", "country_click_rate"]:
            df[col] = df[col].fillna(df[col].median())

    return train, test


def add_user_cluster_features(
    train: pd.DataFrame, test: pd.DataFrame
) -> tuple:
    """
    K-Means user segmentation on search-level features.
    Fit on combined train+test so test users get proper assignments.
    """
    train = train.copy()
    test  = test.copy()

    train["_is_train"] = 1
    test["_is_train"]  = 0
    combined = pd.concat([train, test], axis=0, ignore_index=True)

    user_features = [
        "srch_length_of_stay", "srch_booking_window", "srch_room_count",
        "is_family", "is_solo", "is_couple", "is_group",
        "total_people", "people_per_room",
        "is_long_stay", "is_weekend_trip", "is_last_minute", "is_planned",
        "has_hist_star", "is_high_end_user", "same_country",
        "search_month", "search_day", "search_hour",
    ]

    user_df = combined[["srch_id"] + user_features].drop_duplicates("srch_id").copy()

    log_cols = [
        "srch_booking_window", "srch_length_of_stay", "total_people", "people_per_room"
    ]
    for col in log_cols:
        user_df[col] = np.log1p(user_df[col])

    user_df[user_features] = (
        user_df[user_features]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(-999)
    )

    X_scaled = StandardScaler().fit_transform(user_df[user_features])

    N_CLUSTERS = 6   # one extra: "family planner" now distinct from "long-stay planner"
    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
    user_df["user_cluster"] = km.fit_predict(X_scaled)

    combined = combined.merge(
        user_df[["srch_id", "user_cluster"]], on="srch_id", how="left"
    )

    # One-hot encode clusters for the model
    for k in range(N_CLUSTERS):
        combined[f"cluster_{k}"] = (combined["user_cluster"] == k).astype(int)

    train_out = combined[combined["_is_train"] == 1].drop(columns=["_is_train"])
    test_out  = combined[combined["_is_train"] == 0].drop(columns=["_is_train"])
    return train_out, test_out


# LOAD & PREPARE DATA

print("Loading data…")
def downcast(df):
    for col in df.select_dtypes("float64").columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes("int64").columns:
        df[col] = df[col].astype("int32")
    return df

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

train = downcast(train)
test  = downcast(test)

print(f"Train: {train.shape} | Test: {test.shape}")

train = assign_relevance(train)

print("Basic feature engineering…")
train = add_basic_features(train)
test  = add_basic_features(test)

print("Property-level aggregation features…")
train, test = add_property_agg_features(train, test)

print("User cluster features…")
train, test = add_user_cluster_features(train, test)

# FEATURE SELECTION

DROP_COLS = {
    "srch_id", "date_time", "prop_id",
    "position", "click_bool", "booking_bool",
    "gross_bookings_usd", "relevance",
    "_is_train",
}

features = [
    col for col in train.columns
    if col not in DROP_COLS
    and col in test.columns
    and train[col].dtype != object
]

print(f"Feature count: {len(features)}")

# ── Clean ────────────────────────────────────────────────────
for df in [train, test]:
    df[features] = df[features].replace([np.inf, -np.inf], np.nan).fillna(-999)


# TRAIN / VALIDATION SPLIT 

print("Splitting…")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(gss.split(train, groups=train["srch_id"]))

train_data = train.iloc[train_idx].sort_values("srch_id")
valid_data = train.iloc[valid_idx].sort_values("srch_id")

X_train, y_train = train_data[features], train_data["relevance"]
X_valid, y_valid = valid_data[features], valid_data["relevance"]

train_group = train_data.groupby("srch_id").size().values
valid_group = valid_data.groupby("srch_id").size().values

print(f"Train rows: {len(X_train)} | Valid rows: {len(X_valid)}")

# LAMBDAMART

print("Training LambdaMART…")

model = lgb.LGBMRanker(
    objective       = "lambdarank",
    metric          = "ndcg",
    ndcg_eval_at    = [5],           
    boosting_type   = "gbdt",

    # ── Capacity ─────────────────────────────────────────────
    n_estimators    = 3000,         
    num_leaves      = 127,          
    max_depth       = -1,            

    # ── Regularisation ───────────────────────────────────────
    learning_rate       = 0.02,      
    min_child_samples   = 50,        
    subsample           = 0.8,
    subsample_freq      = 1,
    colsample_bytree    = 0.8,
    reg_alpha           = 0.05,
    reg_lambda          = 1.5,

    # ── LambdaRank specifics ─────────────────────────────────
    label_gain      = [0, 1, 0, 0, 0, 31],   # gains for labels 0-5 (match NDCG weights)

    random_state    = 42,
    n_jobs          = -1,
    importance_type = "gain",
)

model.fit(
    X_train, y_train,
    group       = train_group,
    eval_set    = [(X_valid, y_valid)],
    eval_group  = [valid_group],
    eval_at     = [5],
    callbacks   = [
        lgb.early_stopping(stopping_rounds=150, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)

best_iter = model.best_iteration_
print(f"\nBest iteration: {best_iter}")

# FEATURE IMPORTANCE

importance = (
    pd.DataFrame({
        "feature":    features,
        "importance": model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance.to_csv("feature_importance.csv", index=False)
print("\nTop 20 features:")
print(importance.head(20).to_string(index=False))

# PREDICT & SUBMIT

print("\nPredicting test set…")
test_sorted = test.sort_values("srch_id").copy()
test_sorted["score"] = model.predict(test_sorted[features])

submission = (
    test_sorted
    .sort_values(["srch_id", "score"], ascending=[True, False])
    [["srch_id", "prop_id"]]
)

submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Submission saved → {SUBMISSION_PATH}")
print(f"Rows: {len(submission):,} | Unique queries: {submission['srch_id'].nunique():,}")


Loading data…
Train: (4958347, 54) | Test: (4959183, 50)
Basic feature engineering…
Property-level aggregation features…
User cluster features…
Feature count: 100
Splitting…
Train rows: 3966682 | Valid rows: 991665
Training LambdaMART…


/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.799854 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9208
[LightGBM] [Info] Number of data points in the train set: 3966682, number of used features: 100
Training until validation scores don't improve for 150 rounds
[100]	valid_0's ndcg@5: 0.487345
[200]	valid_0's ndcg@5: 0.495482
[300]	valid_0's ndcg@5: 0.501171
[400]	valid_0's ndcg@5: 0.504659
[500]	valid_0's ndcg@5: 0.506764
[600]	valid_0's ndcg@5: 0.508026
[700]	valid_0's ndcg@5: 0.508532
[800]	valid_0's ndcg@5: 0.508799
[900]	valid_0's ndcg@5: 0.509816
[1000]	valid_0's ndcg@5: 0.509796
[1100]	valid_0's ndcg@5: 0.510326
[1200]	valid_0's ndcg@5: 0.510405
[1300]	valid_0's ndcg@5: 0.510395
Early stopping, best iteration is:
[1156]	valid_0's ndcg@5: 0.510631

Best iteration: 1156

Top 20 features:
                  feature   importance
 

/usr/local/lib/python3.12/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Submission saved → /kaggle/working/submission.csv
Rows: 4,959,183 | Unique queries: 199,549
